# Z-Ordering e Cache

In [ ]:
import os
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from delta.tables import *

MINIO_ACCESS_KEY = "pkIeKAn4xpjoOgXiHQPw"
MINIO_SECRET_KEY = "JZRBfRszxLZzPeaAQaEXk32JxUKv25DVjUoO06Rk"
DATABASE = "default"
BUCKET_BRONZE = "bronze"
BUCKET_SILVER = "silver"
BUCKET_GOLD = "gold"

In [ ]:
%%time
spark = SparkSession.builder.master("spark://spark-master:7077") \
    .appName("MyAppClass02") \
    .config("spark.eventLog.enabled", "true") \
    .config("spark.eventLog.dir", "file:/home/jovyan/work/spark-logs") \
    .config("spark.history.fs.logDirectory", "file:/home/jovyan/work/spark-logs") \
    .config("log4j.rootCategory", "INFO, console") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "1536m") \
    .config("spark.driver.memory", "1536m") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.storage.memoryFraction", "0.4") \
    .config("spark.shuffle.memoryFraction", "0.5") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "512m") \
    .config("spark.sql.parquet.compression.codec", "gzip") \
    .config("spark.sql.orc.compression.codec", "zlib") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC") \
    .getOrCreate()

In [ ]:
sc = spark.sparkContext
hadoop_conf = sc._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
hadoop_conf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

In [ ]:
spark

## PROJECT 1: HOTEL BOOKING

- [Data source](https://www.kaggle.com/datasets/mojtaba142/hotel-booking)

Let's explore the data, also creating an index using z-Ordering and cache strategy

1. Reading data from raw from **Staging**
2. Write data into bronze
3. Explore columns and make discoveries our Delta table
4. Masking Sensitive Data (PII) with hash
5. Write data into **Silver**
6. Choice the index column (by preference a primary ID column) - **Silver**
7. Making analytics with the data calculating some important metrics, to consume to the Dataviz (Dashboard) later
8. Write data into **Gold**
9. Now, apply cache in the **Gold** table to enhance the performance of every consult

### 1 - READING STAGING (RAW)

In [ ]:
location_raw = f"s3a://staging"
file = "hotel_booking.csv"
data_origen = f"{location_raw}/{file}"

In [ ]:
df = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(data_origen)
df = df.withColumnRenamed("phone-number", "phone_number")

### 2 - Write data into bronze

In [ ]:
table_bronze = "hotel_booking_bronze"
location_bronze = f"s3a://{BUCKET_BRONZE}/delta/{table_bronze}"

In [ ]:
# Writing in Delta format
df.write.format("delta") \
    .mode("overwrite") \
    .save(location_bronze)

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{table_bronze}
    USING DELTA
    LOCATION '{location_bronze}'
""")

### 3 - Explore columns and make discoveries our Delta table

In [ ]:
df_delta = spark.sql(f"SELECT * FROM {DATABASE}.{table_bronze}")

In [ ]:
len(df_delta.columns)

In [ ]:
df_delta.count()

In [ ]:
df_delta.printSchema()

In [ ]:
df_delta.limit(10).show(truncate=False)

In [ ]:
+----------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+---------------+-----+---------------------------+-------------------------+------------------+-----------------------+---------------+----------------------+------------+----------------+
|hotel     |is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type  |adr  |required_car_parking_spaces|total_of_special_requests|reservation_status|reservation_status_date|name           |email                 |phone_number|credit_card     |
+----------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+---------------+-----+---------------------------+-------------------------+------------------+-----------------------+---------------+----------------------+------------+----------------+
|City Hotel|0          |4        |2017             |January           |2                       |10                       |0                      |2                   |1     |0.0     |0     |BB  |ITA    |Groups        |TA/TO               |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |null |405.0  |0                   |Transient-Party|70.0 |0                          |0                        |Check-Out         |2017-01-12             |Eric Davis     |Davis.Eric@yahoo.com  |905-427-4517|************7927|
|City Hotel|0          |4        |2017             |January           |2                       |10                       |0                      |2                   |1     |0.0     |0     |BB  |ESP    |Groups        |TA/TO               |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |null |405.0  |0                   |Transient-Party|70.0 |0                          |0                        |Check-Out         |2017-01-12             |Shane Sanchez  |Shane_S17@aol.com     |490-675-5283|************5088|
|City Hotel|0          |0        |2017             |January           |2                       |9                        |1                      |2                   |1     |0.0     |0     |SC  |USA    |Online TA     |TA/TO               |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |7.0  |null   |0                   |Transient      |53.52|0                          |0                        |Check-Out         |2017-01-12             |Brian Ball     |Brian.B@yandex.com    |878-173-6674|************3236|
|City Hotel|0          |101      |2017             |January           |2                       |9                        |1                      |2                   |2     |0.0     |0     |BB  |GBR    |Online TA     |TA/TO               |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |7.0  |null   |0                   |Transient      |72.07|0                          |1                        |Check-Out         |2017-01-12             |Cindy Taylor   |Cindy.T@aol.com       |203-767-1502|************4725|
|City Hotel|0          |74       |2017             |January           |1                       |7                        |2                      |3                   |1     |0.0     |0     |BB  |USA    |Groups        |TA/TO               |0                |0                     |0                             |D                 |D                 |1              |No Deposit  |330.0|null   |0                   |Transient-Party|67.0 |0                          |2                        |Check-Out         |2017-01-12             |Stephanie Smith|Stephanie.S@gmail.com |350-724-7057|************7039|
|City Hotel|0          |4        |2017             |January           |2                       |10                       |0                      |2                   |1     |0.0     |0     |BB  |ITA    |Groups        |TA/TO               |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |null |405.0  |0                   |Transient-Party|70.0 |0                          |0                        |Check-Out         |2017-01-12             |Laura Shaw     |LShaw@verizon.com     |684-695-4620|************4027|
|City Hotel|0          |0        |2017             |January           |2                       |11                       |0                      |1                   |2     |0.0     |0     |BB  |GBR    |Direct        |Direct              |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |null |null   |0                   |Transient      |90.0 |0                          |0                        |Check-Out         |2017-01-12             |Jesse Porter   |JessePorter61@mail.com|155-857-3216|************6431|
|City Hotel|0          |4        |2017             |January           |2                       |10                       |0                      |2                   |1     |0.0     |0     |BB  |PRT    |Groups        |TA/TO               |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |null |405.0  |0                   |Transient-Party|79.0 |0                          |0                        |Check-Out         |2017-01-12             |Robert Moore   |Robert_M@outlook.com  |459-685-2063|************2499|
|City Hotel|0          |0        |2017             |January           |3                       |17                       |0                      |1                   |1     |0.0     |0     |BB  |PRT    |Corporate     |Corporate           |1                |0                     |1                             |A                 |A                 |0              |No Deposit  |null |405.0  |0                   |Transient      |80.0 |0                          |1                        |Check-Out         |2017-01-18             |Cheryl Russell |Cheryl_R@yandex.com   |345-356-4097|************4411|
|City Hotel|0          |4        |2017             |January           |2                       |10                       |0                      |2                   |1     |0.0     |0     |BB  |ITA    |Groups        |TA/TO               |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |null |405.0  |0                   |Transient-Party|70.0 |0                          |0                        |Check-Out         |2017-01-12             |Alison Obrien  |AObrien@att.com       |991-066-4661|************1649|
+----------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+---------------+-----+---------------------------+-------------------------+------------------+-----------------------+---------------+----------------------+------------+----------------+

In [ ]:
df_delta.describe().show()

In [ ]:
+-------+------------+-------------------+------------------+------------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------------------+-------------------+--------------------+---------+-------+--------------+--------------------+-------------------+----------------------+------------------------------+------------------+------------------+-------------------+------------+-----------------+------------------+--------------------+---------------+------------------+---------------------------+-------------------------+------------------+-------------+--------------------+------------+----------------+
|summary|       hotel|        is_canceled|         lead_time| arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|            adults|           children|              babies|     meal|country|market_segment|distribution_channel|  is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|    booking_changes|deposit_type|            agent|           company|days_in_waiting_list|  customer_type|               adr|required_car_parking_spaces|total_of_special_requests|reservation_status|         name|               email|phone_number|     credit_card|
+-------+------------+-------------------+------------------+------------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------------------+-------------------+--------------------+---------+-------+--------------+--------------------+-------------------+----------------------+------------------------------+------------------+------------------+-------------------+------------+-----------------+------------------+--------------------+---------------+------------------+---------------------------+-------------------------+------------------+-------------+--------------------+------------+----------------+
|  count|      119390|             119390|            119390|            119390|            119390|                  119390|                   119390|                 119390|              119390|            119390|             119386|              119390|   119390| 118902|        119390|              119390|             119390|                119390|                        119390|            119390|            119390|             119390|      119390|           103050|              6797|              119390|         119390|            119390|                     119390|                   119390|            119390|       119390|              119390|      119390|          119390|
|   mean|        null|0.37041628277075134|104.01141636652986| 2016.156554150264|              null|       27.16517296255968|       15.798241058715135|     0.9275986263506156|   2.500301532791691|1.8564033838679956|0.10388990333874994|0.007948739425412514|     null|   null|          null|                null|0.03191222045397437|   0.08711784906608594|           0.13709690928888515|              null|              null|0.22112404724013737|        null|86.69338185346919|189.26673532440782|   2.321149174972778|           null|101.83112153446545|        0.06251779881062065|       0.5713627607002262|              null|         null|                null|        null|            null|
| stddev|        null|0.48291822659259864|106.86309704798799|0.7074759445206212|              null|      13.605138355497672|         8.78082947057835|     0.9986134945978757|  1.9082856150479122| 0.579260998832754|0.39856144478644073| 0.09743619130126457|     null|   null|          null|                null|0.17576714541065627|    0.8443363841545108|            1.4974368477076767|              null|              null| 0.6523055726747705|        null|110.7745476429514| 131.6550146385122|    17.5947208787762|           null| 50.53579028554869|         0.2452911474674927|        0.792798422809413|              null|         null|                null|        null|            null|
|    min|  City Hotel|                  0|                 0|              2015|             April|                       1|                        1|                      0|                   0|                 0|                0.0|                   0|       BB|    ABW|      Aviation|           Corporate|                  0|                     0|                             0|                 A|                 A|                  0|  No Deposit|              1.0|               6.0|                   0|       Contract|             -6.38|                          0|                        0|          Canceled|Aaron Acevedo|AAcevedo17@outloo...|100-009-1307|************1000|
|    max|Resort Hotel|                  1|               737|              2017|         September|                      53|                       31|                     19|                  50|                55|               10.0|                  10|Undefined|    ZWE|     Undefined|           Undefined|                  1|                    26|                            72|                 P|                 P|                 21|  Refundable|            535.0|             543.0|                 391|Transient-Party|            5400.0|                          8|                        5|           No-Show|       Zoe Wu|Zuniga_Thomas@out...|999-999-0072|************9999|
+-------+------------+-------------------+------------------+------------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------------------+-------------------+--------------------+---------+-------+--------------+--------------------+-------------------+----------------------+------------------------------+------------------+------------------+-------------------+------------+-----------------+------------------+--------------------+---------------+------------------+---------------------------+-------------------------+------------------+-------------+--------------------+------------+----------------+

### 4 - Masking Sensitive Data (PII) with hash
The query selects only the columns relevant for analysis, masking sensitive personal information such as name, email, phone number, and credit card.

In [ ]:
# Masking a column 'name' using SHA-256
df_delta = df_delta.withColumn('name', F.sha2(F.col('name'), 256))

# Masking a column 'phone-number' using SHA-256
df_delta = df_delta.withColumn('phone_number', F.sha2(F.col('phone_number'), 256))

# Masking a column 'credit_card' using SHA-256
df_delta = df_delta.withColumn('credit_card', F.sha2(F.col('credit_card'), 256))

### 5 - Write data into silver

In [ ]:
table_silver = "hotel_booking_silver"
location_silver = f"s3a://{BUCKET_SILVER}/delta/{table_silver}"

In [ ]:
# Writing in Delta format
df_delta.write.format("delta") \
    .mode("overwrite") \
    .save(location_silver)

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{table_silver}
    USING DELTA
    LOCATION '{location_silver}'
""")

In [ ]:
spark.sql(f"SELECT * FROM {DATABASE}.{table_silver} LIMIT 10").show(truncate=False)

In [ ]:
+----------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+---------------+-----+---------------------------+-------------------------+------------------+-----------------------+----------------------------------------------------------------+----------------------+----------------------------------------------------------------+----------------------------------------------------------------+
|hotel     |is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type  |adr  |required_car_parking_spaces|total_of_special_requests|reservation_status|reservation_status_date|name                                                            |email                 |phone_number                                                    |credit_card                                                     |
+----------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+---------------+-----+---------------------------+-------------------------+------------------+-----------------------+----------------------------------------------------------------+----------------------+----------------------------------------------------------------+----------------------------------------------------------------+
|City Hotel|0          |4        |2017             |January           |2                       |10                       |0                      |2                   |1     |0.0     |0     |BB  |ITA    |Groups        |TA/TO               |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |null |405.0  |0                   |Transient-Party|70.0 |0                          |0                        |Check-Out         |2017-01-12             |c355ab8a0c05fae3e5f7c6566024572df6608850165d99cf88b59cefd28888e5|Davis.Eric@yahoo.com  |1b92c9c81c7011f15317883d60e32f79b0fe19869e578cdf93d63b73386a0e67|676ec5d558364324a125868489409846dd159901c62c1cf2dcef029b08c6d9d4|
|City Hotel|0          |4        |2017             |January           |2                       |10                       |0                      |2                   |1     |0.0     |0     |BB  |ESP    |Groups        |TA/TO               |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |null |405.0  |0                   |Transient-Party|70.0 |0                          |0                        |Check-Out         |2017-01-12             |54ca51a105d64caa6e76b53bfdadc0ad6f605ed565302644e109d5e8f0119d02|Shane_S17@aol.com     |47549c527a058a2ecb9275879345185e6a48f40b86e30b224cccd5d574423d2b|40c84ee88602ff77c181e9dd4d6a065a15ab79c8964bb8ae0864940203409718|
|City Hotel|0          |0        |2017             |January           |2                       |9                        |1                      |2                   |1     |0.0     |0     |SC  |USA    |Online TA     |TA/TO               |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |7.0  |null   |0                   |Transient      |53.52|0                          |0                        |Check-Out         |2017-01-12             |28a18b9a0bb4e159e5bb38715d30dcb938cdef11c258764503d920740a239c87|Brian.B@yandex.com    |2ba10ebf3f7cba04da6eece019fdb9d8c353225bf2469f5ee0bffbbd988937a9|3dfc1d457d3618cdc533448cc85b88466bb83a662e3ef229f3267c9c0f41f58d|
|City Hotel|0          |101      |2017             |January           |2                       |9                        |1                      |2                   |2     |0.0     |0     |BB  |GBR    |Online TA     |TA/TO               |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |7.0  |null   |0                   |Transient      |72.07|0                          |1                        |Check-Out         |2017-01-12             |0de1d3c6802346c4042288d781298139ab09230a88f4406a54caa7e7254baaf9|Cindy.T@aol.com       |c7ad9d2b6abfd64f863e1595f6cd498ebe15f4206706cdc2ece2690aad272e91|bbfde9392e496d9b89f51e771715781d546aeb5430200fe76f2b8f52c5741e75|
|City Hotel|0          |74       |2017             |January           |1                       |7                        |2                      |3                   |1     |0.0     |0     |BB  |USA    |Groups        |TA/TO               |0                |0                     |0                             |D                 |D                 |1              |No Deposit  |330.0|null   |0                   |Transient-Party|67.0 |0                          |2                        |Check-Out         |2017-01-12             |c274d0af13eb96511b4afe6af3cdb416b3187f042ca4a9f257834967487d6720|Stephanie.S@gmail.com |4b6a35d7fec57369cb6ebe93638241f337d63484da0b1820fba355f0660534aa|f2a70eb9e290ab928173915275e6a7c06835c492f8a442f3b7ee72ee175de46b|
|City Hotel|0          |4        |2017             |January           |2                       |10                       |0                      |2                   |1     |0.0     |0     |BB  |ITA    |Groups        |TA/TO               |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |null |405.0  |0                   |Transient-Party|70.0 |0                          |0                        |Check-Out         |2017-01-12             |441765fe924e75cc283767d65c29a09fb1b1b8fc774e720d414c50a83f57a7dd|LShaw@verizon.com     |15a492adbe500d5af5540fff012f71d7ad0d8be55b4290d4f79d9b0292e645bb|a4db3eefd167f83dcb853d7db2439fa7042e0df4b26dde26aac1f0a541134bc5|
|City Hotel|0          |0        |2017             |January           |2                       |11                       |0                      |1                   |2     |0.0     |0     |BB  |GBR    |Direct        |Direct              |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |null |null   |0                   |Transient      |90.0 |0                          |0                        |Check-Out         |2017-01-12             |30ad105f4e4cb347dcee086fa19a12ba37e380e67e630f7ef54fe05739d89d33|JessePorter61@mail.com|2fa599146a4bd7b9c4b3ba2c1b22faa6c0d0bbd93de1e8f12a1f1fbed12cdcdb|930b560f29ed611ebb16b1f7c4c51004ebeb41a7b5492e3acabc8c7139cf0118|
|City Hotel|0          |4        |2017             |January           |2                       |10                       |0                      |2                   |1     |0.0     |0     |BB  |PRT    |Groups        |TA/TO               |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |null |405.0  |0                   |Transient-Party|79.0 |0                          |0                        |Check-Out         |2017-01-12             |e105c89aed7424b89bba9e36e9616fbda15257eb40957546b48b49d33e8be7f0|Robert_M@outlook.com  |936cde36184b7fb298d36ef25ddc80895ecf490820c640a7a0705e589625633e|b162c87abda505516981c3805da5ab0aed2acf2fbb303e05ae38d2f9eca9c44c|
|City Hotel|0          |0        |2017             |January           |3                       |17                       |0                      |1                   |1     |0.0     |0     |BB  |PRT    |Corporate     |Corporate           |1                |0                     |1                             |A                 |A                 |0              |No Deposit  |null |405.0  |0                   |Transient      |80.0 |0                          |1                        |Check-Out         |2017-01-18             |6a6a54640272fdc1d67a9364d412812cd80123b4e83ce5a68a57f55374a137ac|Cheryl_R@yandex.com   |eee70afbd5ec1d06c7523a9f8da5f95dfb9223313f808e4bacbeda99e149f15e|460726152abee794c0703774e03865892ecbfb45fb0b43879311c7e77339ab8d|
|City Hotel|0          |4        |2017             |January           |2                       |10                       |0                      |2                   |1     |0.0     |0     |BB  |ITA    |Groups        |TA/TO               |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |null |405.0  |0                   |Transient-Party|70.0 |0                          |0                        |Check-Out         |2017-01-12             |945d120e9ee26f6368ca68a2f6959f22c00c44ebe07a37e04698bb5c562aa461|AObrien@att.com       |c003da7959cbd0c39359d467618e750b40117babd79e71c53984215cd9a45d79|c5a3a515b2f25500e3c44058dd4deb7d047d9d8b0ce759dd7066c5255e31e2cf|
+----------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+---------------+-----+---------------------------+-------------------------+------------------+-----------------------+----------------------------------------------------------------+----------------------+----------------------------------------------------------------+----------------------------------------------------------------+

### 6 - Choice the index column (by preference a primary ID column) - Silver

How to choose the right column for Z-ordering?

1. Frequently used in filters and joins.
2. With high or moderate cardinality: Columns with many distinct values are excellent candidates. Z-Ordering organizes the data so that similar values are physically close, improving the performance of data skipping.
3. With high cardinality, but with almost unique values, it is not recommended.
4. Not used in partitioning.
5. Do not use PII column.
6. It might be defined 1 and max 5 columns.


In [ ]:
spark.sql(f"""
    SELECT
      COUNT(DISTINCT name) AS unique_name_values,
      COUNT(DISTINCT email) AS unique_email_values,
      COUNT(DISTINCT credit_card) AS unique_credit_card_values,
      COUNT(DISTINCT hotel) AS unique_hotel_values,
      COUNT(DISTINCT is_canceled) AS unique_is_canceled_values,
      COUNT(DISTINCT lead_time) AS unique_lead_time_values,
      COUNT(DISTINCT arrival_date_year) AS unique_arrival_date_year_values,
      COUNT(DISTINCT arrival_date_month) AS unique_arrival_date_month_values,
      COUNT(DISTINCT arrival_date_week_number) AS unique_arrival_date_week_number_values,
      COUNT(DISTINCT arrival_date_day_of_month) AS unique_arrival_date_day_of_month_values,
      COUNT(DISTINCT stays_in_weekend_nights) AS unique_stays_in_weekend_nights_values,
      COUNT(DISTINCT stays_in_week_nights) AS unique_stays_in_week_nights_values,
      COUNT(DISTINCT adults) AS unique_adults_values,
      COUNT(DISTINCT children) AS unique_children_values,
      COUNT(DISTINCT babies) AS unique_babies_values,
      COUNT(DISTINCT meal) AS unique_meal_values,
      COUNT(DISTINCT country) AS unique_country_values,
      COUNT(DISTINCT market_segment) AS unique_market_segment_values,
      COUNT(DISTINCT distribution_channel) AS unique_distribution_channel_values,
      COUNT(DISTINCT is_repeated_guest) AS unique_is_repeated_guest_values,
      COUNT(DISTINCT previous_cancellations) AS unique_previous_cancellations_values,
      COUNT(DISTINCT previous_bookings_not_canceled) AS unique_previous_bookings_not_canceled_values,
      COUNT(DISTINCT reserved_room_type) AS unique_reserved_room_type_values,
      COUNT(DISTINCT assigned_room_type) AS unique_assigned_room_type_values,
      COUNT(DISTINCT booking_changes) AS unique_booking_changes_values,
      COUNT(DISTINCT deposit_type) AS unique_deposit_type_values,
      COUNT(DISTINCT agent) AS unique_agent_values,
      COUNT(DISTINCT company) AS unique_company_values,
      COUNT(DISTINCT days_in_waiting_list) AS unique_days_in_waiting_list_values,
      COUNT(DISTINCT customer_type) AS unique_customer_type_values,
      COUNT(DISTINCT adr) AS unique_adr_values,
      COUNT(DISTINCT required_car_parking_spaces) AS unique_required_car_parking_spaces_values,
      COUNT(DISTINCT total_of_special_requests) AS unique_total_of_special_requests_values,
      COUNT(DISTINCT reservation_status) AS unique_reservation_status_values,
      COUNT(DISTINCT reservation_status_date) AS unique_reservation_status_date_values
    FROM {DATABASE}.{table_silver};
""").show(truncate=False)

In [ ]:
+------------------+-------------------+-------------------------+-------------------+-------------------------+-----------------------+-------------------------------+--------------------------------+--------------------------------------+---------------------------------------+-------------------------------------+----------------------------------+--------------------+----------------------+--------------------+------------------+---------------------+----------------------------+----------------------------------+-------------------------------+------------------------------------+--------------------------------------------+--------------------------------+--------------------------------+-----------------------------+--------------------------+-------------------+---------------------+----------------------------------+---------------------------+-----------------+-----------------------------------------+---------------------------------------+--------------------------------+-------------------------------------+
|unique_name_values|unique_email_values|unique_credit_card_values|unique_hotel_values|unique_is_canceled_values|unique_lead_time_values|unique_arrival_date_year_values|unique_arrival_date_month_values|unique_arrival_date_week_number_values|unique_arrival_date_day_of_month_values|unique_stays_in_weekend_nights_values|unique_stays_in_week_nights_values|unique_adults_values|unique_children_values|unique_babies_values|unique_meal_values|unique_country_values|unique_market_segment_values|unique_distribution_channel_values|unique_is_repeated_guest_values|unique_previous_cancellations_values|unique_previous_bookings_not_canceled_values|unique_reserved_room_type_values|unique_assigned_room_type_values|unique_booking_changes_values|unique_deposit_type_values|unique_agent_values|unique_company_values|unique_days_in_waiting_list_values|unique_customer_type_values|unique_adr_values|unique_required_car_parking_spaces_values|unique_total_of_special_requests_values|unique_reservation_status_values|unique_reservation_status_date_values|
+------------------+-------------------+-------------------------+-------------------+-------------------------+-----------------------+-------------------------------+--------------------------------+--------------------------------------+---------------------------------------+-------------------------------------+----------------------------------+--------------------+----------------------+--------------------+------------------+---------------------+----------------------------+----------------------------------+-------------------------------+------------------------------------+--------------------------------------------+--------------------------------+--------------------------------+-----------------------------+--------------------------+-------------------+---------------------+----------------------------------+---------------------------+-----------------+-----------------------------------------+---------------------------------------+--------------------------------+-------------------------------------+
|81503             |115889             |9000                     |2                  |2                        |479                    |3                              |12                              |53                                    |31                                     |17                                   |35                                |14                  |5                     |5                   |5                 |177                  |8                           |5                                 |2                              |15                                  |73                                          |10                              |12                              |21                           |3                         |333                |352                  |128                               |4                          |8879             |5                                        |6                                      |3                               |926                                  |
+------------------+-------------------+-------------------------+-------------------+-------------------------+-----------------------+-------------------------------+--------------------------------+--------------------------------------+---------------------------------------+-------------------------------------+----------------------------------+--------------------+----------------------+--------------------+------------------+---------------------+----------------------------+----------------------------------+-------------------------------+------------------------------------+--------------------------------------------+--------------------------------+--------------------------------+-----------------------------+--------------------------+-------------------+---------------------+----------------------------------+---------------------------+-----------------+-----------------------------------------+---------------------------------------+--------------------------------+-------------------------------------+

In [ ]:
# Executar o comando OPTIMIZE com ZORDER BY
spark.sql(f"""
    OPTIMIZE {DATABASE}.{table_silver}
    ZORDER BY (credit_card)
""")

Delta Lake by default has configured for collecting stats of columns with:

- `delta.stats.stringPrefixLength=32`

In [ ]:
spark.sql(f"SELECT LENGTH(credit_card) as string_lenght FROM {DATABASE}.{table_silver} LIMIT 1").show(truncate=False)

In [ ]:
# Executar o comando OPTIMIZE com ZORDER BY
spark.sql(f"""
    OPTIMIZE {DATABASE}.{table_silver}
    ZORDER BY (country)
""")

In [ ]:
# Create a DeltaTable object
delta_table = DeltaTable.forName(spark, f"{DATABASE}.{table_silver}")
# Get history of table
history_df = delta_table.history()

In [ ]:
(
    history_df.where(F.col("operation") == "OPTIMIZE")
    .select(
        "version", "timestamp", "operation",
        "operationMetrics.numRemovedFiles",
        "operationMetrics.numAddedFiles"
    ).show(truncate=False)
)

### 7 - Making analytics with the data calculating some important metrics, to consume to the Dataviz (Dashboard) later

We will create a GOLD dataset by filtering customers who have canceled their reservations. However, we're not dealing with just any customer, but rather repeat customers who have a history of canceling other reservations multiple times on our platform. From this GOLD dataset, it will be possible to better understand this customer profile and understand the lead time that led to these cancellations.

In [ ]:
spark.sql(f"""
    SELECT
        hotel,
        email,
        lead_time,
        deposit_type,
        customer_type,
        country
    FROM {DATABASE}.{table_silver}
    WHERE
        is_canceled = 1
        AND previous_cancellations > 1
        AND is_repeated_guest = 1
    GROUP BY
        country,
        hotel,
        email,
        lead_time,
        deposit_type,
        customer_type
    ORDER BY
        country
""").show()

In [ ]:
df_gold.show()

## 8 - Write data into Gold

In [ ]:
table_gold = "hotel_booking_gold"
location_gold = f"s3a://{BUCKET_GOLD}/delta/{table_gold}"

In [ ]:
# Writing in Delta format
df_gold.write.format("delta") \
    .mode("overwrite") \
    .save(location_gold)

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{table_gold}
    USING DELTA
    LOCATION '{location_gold}'
""")

## 9 - Now, apply cache in the Gold table to enhance the performance of every consult

In [ ]:
%%time
(
    spark.sql(f"SELECT * FROM {DATABASE}.{table_gold} WHERE reservation_status_date = '2015-12-09'")
    .limit(10)
    .show(truncate=False)
)

In [ ]:
spark.sql(f"CACHE TABLE {DATABASE}.{table_gold}")

In [ ]:
%%time
(
    spark.sql(f"SELECT * FROM {DATABASE}.{table_gold} WHERE reservation_status_date = '2015-12-09'")
    .limit(10)
    .show(truncate=False)
)

In [ ]:
# When finished flushing the cache
spark.sql(f"UNCACHE TABLE {DATABASE}.{table_gold}")

### Show all tables were generated


In [ ]:
spark.sql(f"USE {DATABASE}")

In [ ]:
spark.sql(f"SHOW TABLES").show()

In [ ]:
spark.stop()